# 06 — Modeling Topik (Multilabel): 5 Representasi × 3 Model

**Tujuan:** melatih dan membandingkan 5 representasi teks (TF-IDF, GloVe, FastText, bge-small, CodeBERT) × 3 model (Logistic Regression, Random Forest, MLP PyTorch) untuk klasifikasi **topik multilabel (top-15)**. Ini tugas utama paper.

**Input (tambahkan semua lewat Add Data):** `data_clean.csv`, `labels.csv`, `split.csv`, `glove.npy`, `fasttext.npy`, `sbert.npy`, `codebert.npy`

**Output** (`results/` dan `tables/`):
- `results/topic_results.csv` — satu baris per kombinasi (hasil CV dan test)
- `results/topic_proba/*.npy` — probabilitas OOF dan test tiap kombinasi
- `tables/topic_*.csv` — tabel pivot, CI bootstrap, F1 per label (bahan Bab IV-B)

## Protokol evaluasi (sama dengan keputusan desain)
1. **Test set (20%) tidak disentuh sampai refit akhir.** Semua pemilihan hyperparameter dan threshold memakai data latih (80%) dengan **fold dari `split.csv`**.
2. Untuk tiap konfigurasi hyperparameter: latih di 4 fold, prediksi fold ke-5, ulangi → **prediksi out-of-fold (OOF)**. Konfigurasi dan **threshold global** dipilih dari OOF yang memaksimalkan Macro-F1.
3. Model terbaik di-**refit pada seluruh data latih**, lalu dievaluasi di test set **sekali**.
4. TF-IDF di-fit hanya pada data latih tiap fold; standardisasi (`StandardScaler`) juga di-fit hanya pada data latih tiap fold.

## Penyimpangan sadar dari rancangan (tulis di Bab III-E/F)
- **Grid threshold 0,05–0,95** (bukan 0,20–0,60): bobot kelas (`balanced`, `pos_weight`) menggeser probabilitas, sehingga threshold optimal bisa di luar rentang 0,20–0,60.
- Tuning memakai fold tetap dari `split.csv`, bukan `GridSearchCV` dengan CV baru, agar split identik di semua notebook.
- Pembobotan kelas: LogReg `class_weight='balanced'`; MLP `pos_weight` (rasio negatif/positif, dibatasi maks 10); Random Forest tanpa bobot (threshold yang mengompensasi).

**Estimasi waktu di Kaggle:** 30-90 menit. Hasil disimpan setelah tiap kombinasi, dan sel utama bisa dijalankan ulang untuk **melanjutkan** dari kombinasi yang belum selesai. Aktifkan GPU untuk MLP.

## 1. Import & Konfigurasi

In [1]:
import glob
import itertools
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

QUICK = False        # True = uji cepat (grid kecil, epoch sedikit). Untuk hasil paper WAJIB False.
SEED = 42
THR_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
TFIDF_MAX = 10000
N_ITER_RF = 2 if QUICK else 6
REPS = ['tfidf', 'glove', 'fasttext', 'sbert', 'codebert']
MODELS = ['logreg', 'rf', 'mlp']

ON_KAGGLE = os.path.exists('/kaggle/input')
OUT_DIR = '/kaggle/working' if ON_KAGGLE else '.'
RES_DIR = os.path.join(OUT_DIR, 'results')
PROBA_DIR = os.path.join(RES_DIR, 'topic_proba')
TAB_DIR = os.path.join(OUT_DIR, 'tables')
for d in (RES_DIR, PROBA_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| QUICK:', QUICK)


def find_file(name):
    if os.path.exists(name):
        return name
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} tidak ditemukan (tambahkan Kaggle Dataset-nya)'
    return hits[0]

Device: cuda | QUICK: False


## 2. Load Data, Label, dan Split

In [2]:
df = pd.read_csv(find_file('data_clean.csv'))
labels = pd.read_csv(find_file('labels.csv'))
split = pd.read_csv(find_file('split.csv'))

# Semua file harus sejajar menurut id
assert (df['id'].values == labels['id'].values).all() and (df['id'].values == split['id'].values).all()

topic_cols = [c for c in labels.columns if c.startswith('topic_')]
Y = labels[topic_cols].values.astype(int)
N, L = Y.shape
texts = df['text_clean'].fillna('').values

is_train = (split['set'] == 'train').values
train_idx = np.where(is_train)[0]
test_idx = np.where(~is_train)[0]
fold = split['fold'].values
folds = [(train_idx[fold[train_idx] != k], train_idx[fold[train_idx] == k]) for k in range(5)]

DENSE = {
    'glove': np.load(find_file('glove.npy')),
    'fasttext': np.load(find_file('fasttext.npy')),
    'sbert': np.load(find_file('sbert.npy')),
    'codebert': np.load(find_file('codebert.npy')),
}
for k, v in DENSE.items():
    assert v.shape[0] == N, f'{k}: jumlah baris {v.shape[0]} != {N}'

assert set(train_idx).isdisjoint(set(test_idx))
print(f'N={N}  labels={L}  train={len(train_idx)}  test={len(test_idx)}')
print({k: v.shape for k, v in DENSE.items()})
print('Ukuran fold CV:', [len(ev) for _, ev in folds])

N=3256  labels=15  train=2605  test=651
{'glove': (3256, 300), 'fasttext': (3256, 300), 'sbert': (3256, 384), 'codebert': (3256, 768)}
Ukuran fold CV: [520, 519, 521, 525, 520]


## 3. Metrik dan Baseline Acak

Metrik: **Macro-F1** (utama, semua label sama bobot), **Micro-F1**, **Hamming loss**.

Baseline acak (*stratified random*): tiap label ditebak positif dengan peluang = prevalensinya di data latih. Ini batas bawah kemampuan "menebak"; model harus jauh di atasnya agar klaim "topik lebih mudah daripada difficulty" bermakna (RQ4).

In [3]:
def macro_f1(Yt, Pred):
    return f1_score(Yt, Pred, average='macro', zero_division=0)


def full_metrics(Yt, Pred):
    return {
        'macro_f1': macro_f1(Yt, Pred),
        'micro_f1': f1_score(Yt, Pred, average='micro', zero_division=0),
        'hamming': hamming_loss(Yt, Pred),
    }


def best_threshold(Yt, P):
    scores = [macro_f1(Yt, (P >= t).astype(int)) for t in THR_GRID]
    i = int(np.argmax(scores))
    return float(THR_GRID[i]), float(scores[i])


# Baseline acak, rata-rata 50 pengundian
rng = np.random.default_rng(SEED)
prev = Y[train_idx].mean(axis=0)
Yte = Y[test_idx]
draws = [full_metrics(Yte, (rng.random(Yte.shape) < prev).astype(int)) for _ in range(50)]
baseline = {k: float(np.mean([d[k] for d in draws])) for k in draws[0]}
print('Baseline acak (test):', {k: round(v, 4) for k, v in baseline.items()})

Baseline acak (test): {'macro_f1': 0.1529, 'micro_f1': 0.2644, 'hamming': 0.2228}


## 4. Akses Fitur (tanpa kebocoran)

TF-IDF dan `StandardScaler` di-fit **hanya pada data latih** tiap fold. Hasilnya di-cache agar tidak dihitung ulang untuk tiap konfigurasi. Standardisasi dipakai untuk LogReg dan MLP (Random Forest tidak membutuhkannya). Ini penting terutama untuk CodeBERT, yang vektornya berskala besar.

In [4]:
_cache = {}


def get_xy(rep, tr, ev, key, scale):
    ck = (rep, key, scale)
    if ck in _cache:
        return _cache[ck]
    if rep == 'tfidf':
        vec = TfidfVectorizer(max_features=TFIDF_MAX, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        Xtr = vec.fit_transform(texts[tr])
        Xev = vec.transform(texts[ev])
    else:
        Xtr, Xev = DENSE[rep][tr], DENSE[rep][ev]
        if scale:
            sc = StandardScaler().fit(Xtr)
            Xtr, Xev = sc.transform(Xtr), sc.transform(Xev)
    _cache[ck] = (Xtr, Xev)
    return Xtr, Xev

## 5. Model

Tiga fungsi `fit_*` dengan antarmuka yang sama: terima data latih dan data evaluasi, kembalikan **probabilitas** (n × 15).

- **LogReg:** One-vs-Rest, `class_weight='balanced'`
- **Random Forest:** satu hutan multi-output (mendukung multilabel langsung), `n_jobs=-1`
- **MLP:** Linear(256) → BatchNorm → ReLU → Dropout(0,3) → Linear(15), `BCEWithLogitsLoss` dengan `pos_weight`, optimizer AdamW

In [5]:
def fit_logreg(Xtr, Ytr, Xev, p):
    clf = OneVsRestClassifier(
        LogisticRegression(C=p['C'], class_weight='balanced', max_iter=2000), n_jobs=-1)
    clf.fit(Xtr, Ytr)
    return clf.predict_proba(Xev)


def fit_rf(Xtr, Ytr, Xev, p):
    rf = RandomForestClassifier(
        n_estimators=p['n_estimators'], max_depth=p['max_depth'],
        min_samples_leaf=p['min_samples_leaf'], max_features=p['max_features'],
        n_jobs=-1, random_state=SEED)
    rf.fit(Xtr, Ytr)
    probs = rf.predict_proba(Xev)                    # list, satu array (n, 2) per label
    out = np.zeros((Xev.shape[0], Ytr.shape[1]))
    for j, pj in enumerate(probs):
        classes = list(rf.classes_[j])
        if 1 in classes:
            out[:, j] = pj[:, classes.index(1)]
    return out


def _dense(X):
    return X.toarray() if sparse.issparse(X) else np.asarray(X)


def fit_mlp(Xtr, Ytr, Xev, p):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    Xtr_t = torch.tensor(_dense(Xtr), dtype=torch.float32, device=DEVICE)
    Xev_t = torch.tensor(_dense(Xev), dtype=torch.float32, device=DEVICE)
    Ytr_t = torch.tensor(Ytr, dtype=torch.float32, device=DEVICE)

    model = nn.Sequential(
        nn.Linear(Xtr_t.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(),
        nn.Dropout(0.3), nn.Linear(256, Ytr.shape[1])).to(DEVICE)

    pos = Ytr.sum(axis=0)
    pos_weight = torch.tensor(np.clip((len(Ytr) - pos) / np.maximum(pos, 1), 1, 10),
                              dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=p['lr'], weight_decay=p['weight_decay'])

    n, bs = Xtr_t.shape[0], 64
    g = torch.Generator(device='cpu').manual_seed(SEED)
    for _ in range(p['epochs']):
        model.train()
        perm = torch.randperm(n, generator=g)
        for s in range(0, n, bs):
            b = perm[s:s + bs].to(DEVICE)
            if len(b) < 2:                           # BatchNorm butuh >= 2 sampel
                continue
            opt.zero_grad()
            loss_fn(model(Xtr_t[b]), Ytr_t[b]).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        logits = torch.cat([model(Xev_t[s:s + 512]) for s in range(0, Xev_t.shape[0], 512)])
    return torch.sigmoid(logits).cpu().numpy()


FIT = {'logreg': fit_logreg, 'rf': fit_rf, 'mlp': fit_mlp}

## 6. Ruang Pencarian Hyperparameter

Bahan untuk Bab III-E (*search space*).

| Model | Ruang pencarian |
|---|---|
| LogReg | `C` ∈ {0,1; 1; 10} (grid) |
| Random Forest | `n_estimators` ∈ {200, 400}, `max_depth` ∈ {None, 20, 40}, `min_samples_leaf` ∈ {1, 2}, `max_features` ∈ {sqrt, 0,2}; **6 konfigurasi acak** (random search) |
| MLP | `lr` ∈ {1e-3, 3e-4} × `epochs` ∈ {30, 60}, `weight_decay` = 1e-2 (grid) |

In [6]:
def make_grids():
    if QUICK:
        return {
            'logreg': [{'C': 1.0}],
            'rf': [{'n_estimators': 50, 'max_depth': 20, 'min_samples_leaf': 1, 'max_features': 'sqrt'}],
            'mlp': [{'lr': 1e-3, 'weight_decay': 1e-2, 'epochs': 5}],
        }
    rf_space = list(itertools.product([200, 400], [None, 20, 40], [1, 2], ['sqrt', 0.2]))
    r = np.random.default_rng(SEED)
    pick = r.choice(len(rf_space), size=N_ITER_RF, replace=False)
    return {
        'logreg': [{'C': c} for c in [0.1, 1.0, 10.0]],
        'rf': [dict(zip(['n_estimators', 'max_depth', 'min_samples_leaf', 'max_features'], rf_space[i]))
               for i in pick],
        'mlp': [{'lr': lr, 'weight_decay': 1e-2, 'epochs': e} for lr in [1e-3, 3e-4] for e in [30, 60]],
    }


GRIDS = make_grids()
print({m: len(g) for m, g in GRIDS.items()}, 'konfigurasi per model')

{'logreg': 3, 'rf': 6, 'mlp': 4} konfigurasi per model


## 7. Prosedur untuk Satu Kombinasi

Untuk satu pasangan (representasi, model):
1. Untuk tiap konfigurasi: hitung prediksi OOF di 5 fold latih, cari threshold global terbaik, catat Macro-F1.
2. Pilih konfigurasi dengan Macro-F1 OOF tertinggi.
3. Hitung rata-rata ± std Macro-F1 per fold (pada threshold terpilih).
4. Refit pada seluruh data latih, prediksi test set, terapkan threshold. **Test set dipakai di langkah ini saja.**

In [7]:
def run_combo(rep, model):
    scale = model in ('logreg', 'mlp')
    fit, grid = FIT[model], GRIDS[model]
    t0 = time.time()
    best = None

    for p in grid:
        oof = np.zeros((N, L))
        for k, (tr, ev) in enumerate(folds):
            Xtr, Xev = get_xy(rep, tr, ev, k, scale)
            oof[ev] = fit(Xtr, Y[tr], Xev, p)
        thr, score = best_threshold(Y[train_idx], oof[train_idx])
        if best is None or score > best['score']:
            best = {'p': p, 'thr': thr, 'score': score, 'oof': oof.copy()}

    # Skor per fold pada konfigurasi + threshold terpilih
    fold_scores = [macro_f1(Y[ev], (best['oof'][ev] >= best['thr']).astype(int)) for _, ev in folds]

    # Refit penuh pada data latih -> evaluasi test (sekali)
    Xtr, Xte = get_xy(rep, train_idx, test_idx, 'full', scale)
    P_test = fit(Xtr, Y[train_idx], Xte, best['p'])
    m = full_metrics(Y[test_idx], (P_test >= best['thr']).astype(int))

    np.save(os.path.join(PROBA_DIR, f'{rep}_{model}_test.npy'), P_test)
    np.save(os.path.join(PROBA_DIR, f'{rep}_{model}_oof.npy'), best['oof'][train_idx])

    return {
        'rep': rep, 'model': model,
        'best_params': json.dumps(best['p']), 'threshold': best['thr'],
        'cv_macro_f1_oof': round(best['score'], 4),
        'cv_macro_f1_mean': round(float(np.mean(fold_scores)), 4),
        'cv_macro_f1_std': round(float(np.std(fold_scores)), 4),
        'test_macro_f1': round(m['macro_f1'], 4),
        'test_micro_f1': round(m['micro_f1'], 4),
        'test_hamming': round(m['hamming'], 4),
        'seconds': round(time.time() - t0, 1),
    }

## 8. Jalankan Semua Kombinasi (15)

Hasil ditulis ke `topic_results.csv` setelah **tiap** kombinasi. Jika sesi Kaggle terputus, jalankan ulang sel ini: kombinasi yang sudah selesai dilewati.

In [8]:
RES_PATH = os.path.join(RES_DIR, 'topic_results.csv')
rows, done = [], set()
if os.path.exists(RES_PATH):
    prev_df = pd.read_csv(RES_PATH)
    rows = prev_df.to_dict('records')
    done = set(zip(prev_df['rep'], prev_df['model']))
    print('Melanjutkan, sudah selesai:', len(done))

for rep in REPS:
    for model in MODELS:
        if (rep, model) in done:
            continue
        row = run_combo(rep, model)
        rows.append(row)
        pd.DataFrame(rows).to_csv(RES_PATH, index=False)
        print(f"[{rep:9s} x {model:6s}] CV={row['cv_macro_f1_mean']:.4f}±{row['cv_macro_f1_std']:.4f} "
              f"| test macro-F1={row['test_macro_f1']:.4f} | thr={row['threshold']} | {row['seconds']}s")

res = pd.DataFrame(rows)
res

[tfidf     x logreg] CV=0.5979±0.0098 | test macro-F1=0.6234 | thr=0.4 | 10.4s
[tfidf     x rf    ] CV=0.5744±0.0096 | test macro-F1=0.6102 | thr=0.2 | 725.8s
[tfidf     x mlp   ] CV=0.5812±0.0102 | test macro-F1=0.5991 | thr=0.15 | 87.7s
[glove     x logreg] CV=0.4909±0.0080 | test macro-F1=0.5241 | thr=0.6 | 14.3s
[glove     x rf    ] CV=0.4776±0.0085 | test macro-F1=0.4879 | thr=0.15 | 915.2s
[glove     x mlp   ] CV=0.5350±0.0099 | test macro-F1=0.5612 | thr=0.5 | 53.7s
[fasttext  x logreg] CV=0.5032±0.0023 | test macro-F1=0.5243 | thr=0.5 | 14.3s
[fasttext  x rf    ] CV=0.4884±0.0098 | test macro-F1=0.5065 | thr=0.2 | 866.7s
[fasttext  x mlp   ] CV=0.5407±0.0086 | test macro-F1=0.5656 | thr=0.5 | 57.9s
[sbert     x logreg] CV=0.5421±0.0078 | test macro-F1=0.5776 | thr=0.65 | 13.3s
[sbert     x rf    ] CV=0.5088±0.0126 | test macro-F1=0.5341 | thr=0.15 | 1190.7s
[sbert     x mlp   ] CV=0.5816±0.0091 | test macro-F1=0.5982 | thr=0.5 | 55.7s
[codebert  x logreg] CV=0.5446±0.0058 | tes

,rep,model,best_params,threshold,cv_macro_f1_oof,cv_macro_f1_mean,cv_macro_f1_std,test_macro_f1,test_micro_f1,test_hamming,seconds
0,tfidf,logreg,"{""C"": 10.0}",0.40,0.5983,0.5979,0.0098,0.6234,0.6805,0.1054,10.4
1,tfidf,rf,"{""n_estimators"": 200, ""max_depth"": null, ""min_...",0.20,0.5748,0.5744,0.0096,0.6102,0.6723,0.1108,725.8
2,tfidf,mlp,"{""lr"": 0.0003, ""weight_decay"": 0.01, ""epochs"":...",0.15,0.5814,0.5812,0.0102,0.5991,0.6662,0.1075,87.7
3,glove,logreg,"{""C"": 0.1}",0.60,0.4913,0.4909,0.0080,0.5241,0.5607,0.1600,14.3
4,glove,rf,"{""n_estimators"": 400, ""max_depth"": 40, ""min_sa...",0.15,0.4775,0.4776,0.0085,0.4879,0.5048,0.2445,915.2
5,glove,mlp,"{""lr"": 0.0003, ""weight_decay"": 0.01, ""epochs"":...",0.50,0.5349,0.5350,0.0099,0.5612,0.6020,0.1446,53.7
6,fasttext,logreg,"{""C"": 0.1}",0.50,0.5037,0.5032,0.0023,0.5243,0.5549,0.1794,14.3
7,fasttext,rf,"{""n_estimators"": 200, ""max_depth"": 40, ""min_sa...",0.20,0.4894,0.4884,0.0098,0.5065,0.5712,0.1637,866.7
8,fasttext,mlp,"{""lr"": 0.0003, ""weight_decay"": 0.01, ""epochs"":...",0.50,0.5411,0.5407,0.0086,0.5656,0.6172,0.1337,57.9
9,sbert,logreg,"{""C"": 0.1}",0.65,0.5425,0.5421,0.0078,0.5776,0.6218,0.1272,13.3


## 9. Tabel Hasil (Bab IV-B)

Tabel pivot untuk paper. `Macro-F1 CV` memakai OOF data latih; `test` adalah evaluasi sekali di test set. **Pilih model terbaik berdasarkan CV, bukan test**, supaya klaim di paper tidak bias.

In [9]:
res = pd.read_csv(RES_PATH)
for metric in ['cv_macro_f1_mean', 'test_macro_f1', 'test_micro_f1', 'test_hamming']:
    pv = res.pivot(index='rep', columns='model', values=metric).reindex(REPS)[MODELS]
    pv.to_csv(os.path.join(TAB_DIR, f'topic_pivot_{metric}.csv'))
    print(metric)
    print(pv.round(4).to_string())
    print()

print('Baseline acak (test):', {k: round(v, 4) for k, v in baseline.items()})
best_cv = res.sort_values('cv_macro_f1_mean', ascending=False).iloc[0]
print(f"\nTerbaik menurut CV: {best_cv['rep']} x {best_cv['model']} "
      f"(CV {best_cv['cv_macro_f1_mean']}, test {best_cv['test_macro_f1']})")

cv_macro_f1_mean
model     logreg      rf     mlp
rep                             
tfidf     0.5979  0.5744  0.5812
glove     0.4909  0.4776  0.5350
fasttext  0.5032  0.4884  0.5407
sbert     0.5421  0.5088  0.5816
codebert  0.5446  0.4554  0.5657

test_macro_f1
model     logreg      rf     mlp
rep                             
tfidf     0.6234  0.6102  0.5991
glove     0.5241  0.4879  0.5612
fasttext  0.5243  0.5065  0.5656
sbert     0.5776  0.5341  0.5982
codebert  0.5574  0.4870  0.6059

test_micro_f1
model     logreg      rf     mlp
rep                             
tfidf     0.6805  0.6723  0.6662
glove     0.5607  0.5048  0.6020
fasttext  0.5549  0.5712  0.6172
sbert     0.6218  0.5403  0.6493
codebert  0.6167  0.5253  0.6650

test_hamming
model     logreg      rf     mlp
rep                             
tfidf     0.1054  0.1108  0.1075
glove     0.1600  0.2445  0.1446
fasttext  0.1794  0.1637  0.1337
sbert     0.1272  0.2188  0.1183
codebert  0.1334  0.2254  0.1125

Baseline acak 

## 10. Interval Kepercayaan Bootstrap (test set)

Test set hanya ± 651 soal dan beberapa label punya sekitar 40 positif, jadi selisih kecil antar model bisa berupa noise. Bootstrap atas baris test (500 resampel) memberi interval 95% untuk Macro-F1. **Kalau dua interval banyak tumpang tindih, jangan menyatakan satu model "unggul".**

In [10]:
rng = np.random.default_rng(SEED)
n_te = len(test_idx)
boot_idx = [rng.integers(0, n_te, n_te) for _ in range(500)]
Yte = Y[test_idx]

ci_rows = []
for _, r in res.iterrows():
    P = np.load(os.path.join(PROBA_DIR, f"{r['rep']}_{r['model']}_test.npy"))
    pred = (P >= r['threshold']).astype(int)
    vals = [macro_f1(Yte[b], pred[b]) for b in boot_idx]
    ci_rows.append({'rep': r['rep'], 'model': r['model'],
                    'ci_low': round(float(np.percentile(vals, 2.5)), 4),
                    'ci_high': round(float(np.percentile(vals, 97.5)), 4)})

res_ci = res.merge(pd.DataFrame(ci_rows), on=['rep', 'model'])
res_ci.to_csv(os.path.join(TAB_DIR, 'topic_results_with_ci.csv'), index=False)
res_ci[['rep', 'model', 'cv_macro_f1_mean', 'test_macro_f1', 'ci_low', 'ci_high',
        'test_micro_f1', 'test_hamming']].sort_values('test_macro_f1', ascending=False)

,rep,model,cv_macro_f1_mean,test_macro_f1,ci_low,ci_high,test_micro_f1,test_hamming
0,tfidf,logreg,0.5979,0.6234,0.5970,0.6446,0.6805,0.1054
1,tfidf,rf,0.5744,0.6102,0.5829,0.6325,0.6723,0.1108
14,codebert,mlp,0.5657,0.6059,0.5831,0.6271,0.6650,0.1125
2,tfidf,mlp,0.5812,0.5991,0.5738,0.6216,0.6662,0.1075
11,sbert,mlp,0.5816,0.5982,0.5737,0.6211,0.6493,0.1183
9,sbert,logreg,0.5421,0.5776,0.5561,0.5984,0.6218,0.1272
8,fasttext,mlp,0.5407,0.5656,0.5394,0.5881,0.6172,0.1337
5,glove,mlp,0.5350,0.5612,0.5340,0.5842,0.6020,0.1446
12,codebert,logreg,0.5446,0.5574,0.5357,0.5780,0.6167,0.1334
10,sbert,rf,0.5088,0.5341,0.5087,0.5568,0.5403,0.2188


## 11. F1 per Label

Untuk pembahasan "label mana yang paling sulit" (Bab IV-C). Disimpan untuk **semua** kombinasi; tabel utama memakai kombinasi terbaik menurut CV.

In [11]:
support = Yte.sum(axis=0)
pl = {}
for _, r in res.iterrows():
    P = np.load(os.path.join(PROBA_DIR, f"{r['rep']}_{r['model']}_test.npy"))
    pred = (P >= r['threshold']).astype(int)
    pl[f"{r['rep']}|{r['model']}"] = f1_score(Yte, pred, average=None, zero_division=0)

pl_df = pd.DataFrame(pl, index=[c.replace('topic_', '') for c in topic_cols]).round(3)
pl_df.insert(0, 'support_test', support)
pl_df.to_csv(os.path.join(TAB_DIR, 'topic_per_label_f1_all.csv'))

best_key = f"{best_cv['rep']}|{best_cv['model']}"
best_table = pl_df[['support_test', best_key]].sort_values(best_key)
best_table.to_csv(os.path.join(TAB_DIR, 'topic_per_label_f1_best.csv'))
print('Kombinasi terbaik (CV):', best_key)
best_table

Kombinasi terbaik (CV): tfidf|logreg


,support_test,tfidf|logreg
prefix_sum,49,0.358
binary_search,60,0.426
two_pointers,44,0.438
sorting,92,0.448
greedy,86,0.453
bit_manipulation,52,0.479
hash_table,141,0.492
dynamic_programming,119,0.534
breadth_first_search,41,0.586
math,122,0.682


## Ringkasan untuk paper

Dari notebook ini salin: (1) tabel pivot Macro-F1, Micro-F1, dan Hamming loss, (2) baris baseline acak, (3) interval bootstrap untuk membandingkan model, (4) F1 per label untuk kombinasi terbaik, (5) `best_params` dan `threshold` tiap kombinasi untuk Bab III-E.

**Hati-hati dengan klaim:**
- Jangan menulis model A "mengungguli" B bila interval bootstrap keduanya tumpang tindih banyak.
- Pilih "terbaik" menurut skor CV, bukan test.
- Kalau TF-IDF atau FastText ternyata mengalahkan CodeBERT, itu temuan yang sah. Jelaskan lewat keterbatasan (frozen feature, pemotongan teks, input bahasa alami saja), jangan diubah narasinya.